# Japanese IQ Test

The following puzzle is part of a recruitment test in Japan.

---

A policeman, a convict, a father and his two sons Anton and Bruno, and a mother with her two daughters Cindy and Doris have to cross a river.  On the boat there is only room for two passengers.

During the crossing, the following conditions have to be observed:
* The father is not allowed to be on a shore with one of the daughters 
  if the mother is on the other shore. 
* The mother is not allowed to be on a shore with one of the sons 
  if the father is on the other shore.
* If the criminal is not alone, then the policeman must watch him.

  However the convict can be alone on a shore, as his shackles prevent him from running away. 
* Only the father, the mother, and the policeman are able to steer the boat.

We are going to formulate this problem as a *symbolic transition problem*.
Then, we can solve the problem using the constraint solver `Z3`.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

We will use the following variables to encode the problem:
* `P` equals `1` if the policeman is on the western shore,
* `V` equals `1` if the villain is on the western shore,
* `M` equals `1` if the mother is on the western shore,
* `F` equals `1` if the father is on the western shore,
* `B` is the number of boys on the western shore,
* `G` is the number of girls on the western shore.

The variables `PX`, $\cdots$, `GX` are only used to test the functions defined below.

In [3]:
let P  = Int "P"
let V  = Int "V"
let M  = Int "M"
let F  = Int "F"
let B  = Int "B"
let G  = Int "G"
let PX = Int "PX"
let VX = Int "VX"
let MX = Int "MX"
let FX = Int "FX"
let BX = Int "BX"
let GX = Int "GX"

The function `start` takes the six variables `P`, `V`, `M`, `F`, `B`, and `G` describing a state.  It returns a list of constraints specifying that everybody is on the western shore.

In [4]:
let start (P: IntExpr) (V: IntExpr) (M: IntExpr) (F: IntExpr) (B: IntExpr) (G: IntExpr) : BoolExpr list =
    [ P .= 1; V .= 1; M .= 1; F .= 1; B .= 2; G .= 2 ]

In [5]:
start P V M F B G

[P = 1; V = 1; M = 1; F = 1; B = 2; G = 2]

The function `goal` takes the six variables describing a state.  It returns a list of constraints specifying that everybody is on the eastern shore.

In [6]:
let goal (P: IntExpr) (V: IntExpr) (M: IntExpr) (F: IntExpr) (B: IntExpr) (G: IntExpr) : BoolExpr list =
    [ P .= 0; V .= 0; M .= 0; F .= 0; B .= 0; G .= 0 ]

In [7]:
goal P V M F B G

[P = 0; V = 0; M = 0; F = 0; B = 0; G = 0]

The function `invariant` takes the six variables describing a state.  It returns a list of constraints that have to be true in every state:
* The policeman has to watch the villain (unless the villain is alone):  If the villain and the policeman are on different shores, then all members of the family have to be on the shore of the policeman.  If the policeman is on the western shore, i.e. if $P = 1$, then $M + F + B + G = 6$.  If he is on the eastern shore, then $M + F + B + G = 0$.
* The boys must not be alone with the mother:  If the mother and the father are on different shores, then the boys have to be with the father and the girls have to be with the mother.

In [8]:
let invariant (P: IntExpr) (V: IntExpr) (M: IntExpr) (F: IntExpr) (B: IntExpr) (G: IntExpr) : BoolExpr list =
    [ Implies(V .<> P, M + F + B + G .= 6 * P)
      Implies(M .<> F, And [ B .= 2 * F; G .= 2 * M ]) ]

In [9]:
invariant P V M F B G

[¬(V = P) → M + F + B + G = 6 * P; ¬(M = F) → B = 2 * F ∧ G = 2 * M]

The function `boatOK` takes twelve arguments:
* `Pα` is the number of police men on the left shore before the crossing,
* `Vα` is the number of villains   on the left shore before the crossing,
* `Mα` is the number of mothers    on the left shore before the crossing,
* `Fα` is the number of fathers    on the left shore before the crossing,
* `Bα` is the number of boys       on the left shore before the crossing,
* `Gα` is the number of girls      on the left shore before the crossing,
* `Pβ`, $\cdots$, `Gβ` are the corresponding numbers after the crossing.

It returns a list of constraints that describe a crossing of the boat from the left shore to the right shore.  The constraints ensure that:
* The people move from the left shore to the right shore,
* only the policeman, the father, or the mother can steer the boat, i.e. at least one of them is on the boat,
* there can be at most two persons on the boat.

In [10]:
let boatOK (Pα: IntExpr) (Vα: IntExpr) (Mα: IntExpr) (Fα: IntExpr) (Bα: IntExpr) (Gα: IntExpr)
           (Pβ: IntExpr) (Vβ: IntExpr) (Mβ: IntExpr) (Fβ: IntExpr) (Bβ: IntExpr) (Gβ: IntExpr) : BoolExpr list =
    [ Pα .>= Pβ; Vα .>= Vβ; Mα .>= Mβ; Fα .>= Fβ; Bα .>= Bβ; Gα .>= Gβ
      1 .<= Pα + Mα + Fα - (Pβ + Mβ + Fβ)
      Pα + Vα + Mα + Fα + Bα + Gα - (Pβ + Vβ + Mβ + Fβ + Bβ + Gβ) .<= 2 ]

In [11]:
boatOK P V M F B G PX VX MX FX BX GX

[P ≥ PX; V ≥ VX; M ≥ MX; F ≥ FX; B ≥ BX; G ≥ GX; 1 ≤ P + M + F - (PX + MX + FX); P + V + M + F + B + G - (PX + VX + MX + FX + BX + GX) ≤ 2]

The function `transition` takes the twelve arguments of `boatOK` and the number `i` of the crossing, where $i$ starts counting from $0$.  It returns a list of constraints describing the $i^{\mathrm{th}}$ crossing of the river.  If $i$ is even, the boat crosses from the left shore to the right shore.  Otherwise, it crosses from the right shore to the left shore.  In the second case, we can use `boatOK` with the roles of the states before and after the crossing interchanged.

In [12]:
let transition (Pα: IntExpr) (Vα: IntExpr) (Mα: IntExpr) (Fα: IntExpr) (Bα: IntExpr) (Gα: IntExpr)
               (Pβ: IntExpr) (Vβ: IntExpr) (Mβ: IntExpr) (Fβ: IntExpr) (Bβ: IntExpr) (Gβ: IntExpr)
               (i: int) : BoolExpr list =
    if i % 2 = 0 then
        boatOK Pα Vα Mα Fα Bα Gα Pβ Vβ Mβ Fβ Bβ Gβ
    else
        boatOK Pβ Vβ Mβ Fβ Bβ Gβ Pα Vα Mα Fα Bα Gα

In [13]:
transition P V M F B G PX VX MX FX BX GX 0

[P ≥ PX; V ≥ VX; M ≥ MX; F ≥ FX; B ≥ BX; G ≥ GX; 1 ≤ P + M + F - (PX + MX + FX); P + V + M + F + B + G - (PX + VX + MX + FX + BX + GX) ≤ 2]

In [14]:
transition PX VX MX FX BX GX P V M F B G 1

[P ≥ PX; V ≥ VX; M ≥ MX; F ≥ FX; B ≥ BX; G ≥ GX; 1 ≤ P + M + F - (PX + MX + FX); P + V + M + F + B + G - (PX + VX + MX + FX + BX + GX) ≤ 2]

The function `japaneseCSP` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem with `n` crossings.  If this is possible, `Some Solution` is returned, where `Solution` is a map assigning values to the variables.  Otherwise, `None` is returned.  The Python version of this notebook uses bit vectors of length 5 for the variables.  We use integer variables instead and restrict their values explicitly.

In [15]:
let japaneseCSP (n: int) : Map<string, int> option =
    let S  = newSolver ()
    let vars name = [| for i in 0 .. n -> Int (sprintf "%s%d" name i) |]
    let Ps, Vs, Ms, Fs, Bs, Gs = vars "P", vars "V", vars "M", vars "F", vars "B", vars "G"
    let Cts = [ yield! start Ps[0] Vs[0] Ms[0] Fs[0] Bs[0] Gs[0]
                yield! goal  Ps[n] Vs[n] Ms[n] Fs[n] Bs[n] Gs[n]
                for i in 0 .. n - 1 do
                    yield! invariant  Ps[i] Vs[i] Ms[i] Fs[i] Bs[i] Gs[i]
                    yield! transition Ps[i]   Vs[i]   Ms[i]   Fs[i]   Bs[i]   Gs[i]
                                      Ps[i+1] Vs[i+1] Ms[i+1] Fs[i+1] Bs[i+1] Gs[i+1] i
                    for X in [ Ps; Vs; Ms; Fs; Bs; Gs ] do
                        yield 0 .<= X[i]
                    for X in [ Ps; Vs; Ms; Fs ] do
                        yield X[i] .<= 1
                    yield Bs[i] .<= 2
                    yield Gs[i] .<= 2 ]
    S.Add(Array.ofList Cts)
    if S.Check() = Status.SATISFIABLE then
        let m = S.Model
        Some (Map [ for (name, X) in [ ("P", Ps); ("V", Vs); ("M", Ms); ("F", Fs); ("B", Bs); ("G", Gs) ] do
                      for i in 0 .. n -> (sprintf "%s%d" name i, intValue m X[i]) ])
    else
        None

The function `findSolution` takes no arguments.  It tries $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings together with the solution.

In [16]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match japaneseCSP n with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [17]:
#!time
let n, Solution = findSolution ()

1

3

5

7

9

11

13

15

17

Wall time: 1036.7452ms

In [18]:
n, Solution

(17, map [("B0", 2); ("B1", 2); ("B10", 2); ("B11", 2); ("B12", 2); ("B13", 1); ("B14", 1); ("B15", 0); ("B16", 0); ("B17", 0); ("B2", 2); ("B3", 2); ("B4", 2); ("B5", 2); ("B6", 2); ("B7", 2); ("B8", 2); ("B9", 2); ("F0", 1); ("F1", 1); ("F10", 1); ("F11", 0); ("F12", 1); ("F13", 0); ("F14", 0); ("F15", 0); ("F16", 0); ("F17", 0); ("F2", 1); ("F3", 1); ("F4", 1); ("F5", 1); ("F6", 1); ("F7", 0); ("F8", 1); ("F9", 1); ("G0", 2); ("G1", 2); ("G10", 0); ("G11", 0); ("G12", 0); ("G13", 0); ("G14", 0); ("G15", 0); ("G16", 0); ("G17", 0); ("G2", 2); ("G3", 1); ("G4", 1); ("G5", 0); ("G6", 0); ("G7", 0); ("G8", 0); ("G9", 0); ("M0", 1); ("M1", 1); ("M10", 1); ("M11", 0); ("M12", 0); ("M13", 0); ("M14", 0); ("M15", 0); ("M16", 0); ("M17", 0); ("M2", 1); ("M3", 1); ("M4", 1); ("M5", 0); ("M6", 1); ("M7", 0); ("M8", 0); ("M9", 0); ("P0", 1); ("P1", 0); ("P10", 0); ("P11", 0); ("P12", 0); ("P13", 0); ("P14", 1); ("P15", 0); ("P16", 1); ("P17", 0); ("P2", 1); ("P3", 0); ("P4", 1); ("P5", 1); ("P6

## Auxiliary Code for Pretty Printing

The function `showSolution` takes two arguments:
- `Solution` is a map assigning values to the variables,
- `n` is the number of crossings.

It prints the path that has been found.  We won't discuss the details of this function.

In [19]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    let get name i = Solution[sprintf "%s%d" name i]
    let persons P V M F B G = rep P "👮" + rep V "🦹" + rep M "👩" + rep F "👨🏿" + rep B "👦" + rep G "👧🏾"
    for i in 0 .. n do
        let P, V, M, F, B, G = get "P" i, get "V" i, get "M" i, get "F" i, get "B" i, get "G" i
        printfn "%s%s%s" (persons P V M F B G) (String.replicate 42 " ") (persons (1-P) (1-V) (1-M) (1-F) (2-B) (2-G))
        if i < n then
            let sign = if i % 2 = 0 then 1 else -1
            let crossing name = sign * (get name i - get name (i+1))
            let arrows = if i % 2 = 0 then ">>>" else "<<<"
            printfn "%s%s%s%s" (String.replicate 24 " ") arrows
                    (persons (crossing "P") (crossing "V") (crossing "M") (crossing "F") (crossing "B") (crossing "G")) arrows

In [20]:
showSolution Solution n

👮🦹👩👨🏿👦👦👧🏾👧🏾

>>>

👮🦹

>>>

👩👨🏿👦👦👧🏾👧🏾

...

👩👨🏿👦👦👧🏾👧🏾

>>>

👮🦹

>>>

👮🦹👩👨🏿👦👦👧🏾👧🏾